# 03 — Exact NHT variance and variance-estimator comparison

This notebook computes the exact design variance of the Narain--Horvitz--Thompson estimator and compares the selected variance estimators reported in the paper.

**Main outputs.** A table containing the true NHT variance, estimator averages, and relative bias for selected saved designs and response functions.

**How to use.** Run the setup cells, load the saved designs, then run the analysis cells. Optional cells at the end produce rounded displays and LaTeX tables.


## Reproducibility note

Outputs and execution counts were cleared intentionally, so the notebook opens without stale errors. Run the cells section by section after confirming that the local package and data paths are available.

In [21]:
# Purpose: locate the simulation working directory without using a user-specific absolute path.
from pathlib import Path
import os

def set_working_directory(prefer="simulations"):
    """Move to the repository's simulation folder if it can be found.

    The notebooks were prepared from the graphical-sampling repository. This
    helper keeps them portable and ensures that, if the notebook is launched
    from simulations/jupyters, the working directory is set to simulations.
    """
    cwd = Path.cwd().resolve()
    markers = ["results", "best_designs", "populations", "data", "maps"]

    candidates = []

    # Most important case: notebook opened from simulations/jupyters
    if cwd.name == "jupyters" and cwd.parent.name == prefer:
        candidates.append(cwd.parent)

    # If already somewhere inside simulations, return the simulations folder
    for path in [cwd, *cwd.parents]:
        if path.name == prefer:
            candidates.append(path)
            break

    # Other common portable locations
    candidates.extend([
        cwd / prefer,
        cwd.parent / prefer,
        Path.home() / "projects" / "graphical-sampling" / prefer,
        Path.home() / "graphical-sampling" / prefer,
    ])

    for path in candidates:
        if path.exists() and path.is_dir() and any((path / marker).exists() for marker in markers):
            os.chdir(path)
            print(f"Working directory: {Path.cwd()}")
            return Path.cwd()

    print(f"Working directory unchanged: {Path.cwd()}")
    return Path.cwd()

WORKDIR = set_working_directory()

Working directory: /home/bardia/projects/graphical-sampling/simulations


In [22]:
# Purpose: import packages and configure helper functions.
import os
# Path is handled by set_working_directory().
set_working_directory()

Working directory: /home/bardia/projects/graphical-sampling/simulations


PosixPath('/home/bardia/projects/graphical-sampling/simulations')

In [23]:
# Purpose: import packages and configure helper functions.
# =========================================================
# CELL 1
# SETTINGS AND IMPORTS
# =========================================================

import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Project root
# ---------------------------------------------------------
#
# If needed, uncomment and edit:

print("Current working directory:")
print(os.getcwd())

# ---------------------------------------------------------
# Main switches
# ---------------------------------------------------------

INCLUDE_SWISS = False       # Swiss was slow; keep False for the main paper table
SAVE_PARTIAL = True
OUTPUT_DIR = Path("variance_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------
# Estimator choices
# ---------------------------------------------------------

LM_SIMPLE_Q = 8
LM_SIMPLE_INCLUDE_SELF = False
LM_SIMPLE_WEIGHT_METHOD = "equal"

TILLE_J = 3
TILLE_DISTANCE_POWER = 1.0
TILLE_PI_POWER = 1.0
TILLE_INCLUDE_SELF = True
TILLE_SINKHORN_ITER = 200

HYBRID_ALPHA = 0.35

print("Settings loaded.")

Current working directory:
/home/bardia/projects/graphical-sampling/simulations
Settings loaded.


In [24]:
# Purpose: define the Robertson benchmark response surfaces.
# =========================================================
# CELL 2
# ROBERTSON VARIABLES
# =========================================================

def robertson_variables(x1, x2, positive_shift=True):
    """Robertson artificial response functions.

    Note:
    peak and bird are shifted by +7 and +6, respectively.
    """

    corrugated = (
        3 * (x1 + x2)
        +
        np.sin(6 * (x1 + x2))
    )

    peak = (
        3 * (4 - 6*x1)**2
        *
        np.exp(
            -(6*x1 - 3)**2
            -
            (6*x2 - 2)**2
        )
        -
        10
        *
        (
            0.2*(6*x1 - 3)
            -
            (6*x1 - 3)**3
            -
            (6*x2 - 3)**5
        )
        *
        np.exp(
            -(6*x1 - 3)**2
            -
            (6*x2 - 3)**2
        )
        -
        (1/3)
        *
        np.exp(
            -(6*x1 - 2)**2
            -
            (6*x2 - 3)**2
        )
    )

    bird = (
        1/20
        *
        (
            (12*x1 - 12*x2)**2
            +
            np.exp(
                (1 - np.sin(12*x1 - 6))**2
            )
            *
            np.cos(12*x2 - 6)
            +
            np.exp(
                (1 - np.cos(12*x2 - 6))**2
            )
            *
            np.sin(12*x1 - 6)
        )
    )

    if positive_shift:
        peak = peak + 7
        bird = bird + 6

    return {
        "corrugated": corrugated,
        "plane": corrugated,
        "peak": peak,
        "bird": bird,
    }

print("Robertson variables defined.")

Robertson variables defined.


In [25]:
# Purpose: load and prepare the finite populations used in the experiments.
# =========================================================
# CELL 3
# LOAD POPULATIONS
# =========================================================

from package_sampling.utils import inclusion_probabilities

POP_FILES = [
    "meuse.csv",
    "swiss.csv",
    "AggregatedPop1000.csv",
    "RegularPop1000.csv",
]

def popus(n, path="populations/real"):
    """Load all populations for a given sample size n.

    The inclusion probabilities depend on n, so this function is
    intentionally called again for each design.
    """

    dfs = {}

    for f in POP_FILES:

        full_path = os.path.join(path, f)

        if not os.path.exists(full_path):
            print(f"Warning: {full_path} not found.")
            continue

        raw = pd.read_csv(full_path)
        N_pop = len(raw)
        name = f.replace(".csv", "")

        pik_ep = np.repeat(n / N_pop, N_pop)

        # -------------------------------------------------
        # Robertson populations
        # -------------------------------------------------
        if f in ["AggregatedPop1000.csv", "RegularPop1000.csv"]:

            coords = raw.to_numpy(float)

            x1 = coords[:, 0]
            x2 = coords[:, 1]

            vars_rob = robertson_variables(x1, x2)

            dfs[f"df_{name}"] = pd.DataFrame({
                "coord_x": x1,
                "coord_y": x2,
                "pik_ep": pik_ep,
                "z": vars_rob["corrugated"],
                "plane": vars_rob["plane"],
                "peak": vars_rob["peak"],
                "bird": vars_rob["bird"],
            })

        # -------------------------------------------------
        # Meuse
        # -------------------------------------------------
        elif f == "meuse.csv":

            coords = raw[["x", "y"]].to_numpy(float)

            pik_up = inclusion_probabilities(
                raw["copper"].to_numpy(float).copy(),
                n
            )

            dfs[f"df_{name}"] = pd.DataFrame({
                "coord_x": coords[:, 0],
                "coord_y": coords[:, 1],
                "pik_ep": pik_ep,
                "pik_up": pik_up,
                "z": raw["cadmium"].to_numpy(float),
            })

        # -------------------------------------------------
        # Swiss
        # -------------------------------------------------
        elif f == "swiss.csv":

            coords = raw[["x", "y"]].to_numpy(float)

            area = raw["AREA"].to_numpy(float).clip(5, 100)

            pik_up = inclusion_probabilities(
                area.copy(),
                n
            )

            dfs[f"df_{name}"] = pd.DataFrame({
                "coord_x": coords[:, 0],
                "coord_y": coords[:, 1],
                "pik_ep": pik_ep,
                "pik_up": pik_up,
                "z": raw["AREA_A"].to_numpy(float),
            })

    return dfs


# quick check
p50 = popus(50)

print("Loaded populations for n=50:")
for key, df in p50.items():
    print(key, df.shape)

Loaded populations for n=50:
df_meuse (155, 5)
df_swiss (959, 5)
df_AggregatedPop1000 (1000, 7)
df_RegularPop1000 (1000, 7)


In [26]:
# Purpose: run the next step in the notebook workflow.
# =========================================================
# CELL 4
# LOAD ALL DESIGNS
# =========================================================

root = Path("best_designs")

designs = {}
failed_files = []

def simplify_name(population, stem):

    if stem.startswith("best_design"):
        method = "best"
    elif stem.startswith("initial_design"):
        method = "initial"
    else:
        method = "unknown"

    # -----------------------------------------------------
    # Meuse / Swiss
    # -----------------------------------------------------
    if population in ["meuse", "swiss"]:

        parts = stem.split("_")

        n = parts[-3]
        mode = parts[-1]

        return f"{population}_{method}_{n}_{mode}"

    # -----------------------------------------------------
    # Robertson populations
    # -----------------------------------------------------
    if population == "robertson":

        if "Aggregated" in stem:
            pop = "aggregated"
        elif "Regular" in stem:
            pop = "regular"
        else:
            pop = "unknown"

        if "C20" in stem:
            n = "20"
        elif "C50" in stem:
            n = "50"
        else:
            n = "?"

        mode = "ep"

        return f"{pop}_{method}_{n}_{mode}"

    return f"{population}_{stem}"


for pop_folder in root.iterdir():

    if not pop_folder.is_dir():
        continue

    population = pop_folder.name

    for file in pop_folder.glob("*.pkl"):

        try:

            with open(file, "rb") as f:
                obj = pickle.load(f)

            # supports either direct Design or GBFS object
            design = obj.best_design if hasattr(obj, "best_design") else obj

            short_name = simplify_name(
                population,
                file.stem
            )

            designs[short_name] = design

        except Exception as e:

            failed_files.append((str(file), str(e)))


print("\nLoaded designs:")
for k in sorted(designs):
    print(k)

print("\nTotal loaded:", len(designs))

if failed_files:
    print("\nFailed files:")
    for f, err in failed_files:
        print(f)
        print(err)
        print("-" * 60)


Loaded designs:
aggregated_best_20_ep
aggregated_best_50_ep
aggregated_best_?_ep
aggregated_initial_20_ep
aggregated_initial_50_ep
aggregated_initial_?_ep
meuse_best_10_ep
meuse_best_10_up
meuse_best_20_ep
meuse_best_20_up
meuse_best_5_ep
meuse_best_5_up
meuse_initial_10_ep
meuse_initial_10_up
meuse_initial_20_ep
meuse_initial_20_up
meuse_initial_5_ep
meuse_initial_5_up
regular_best_20_ep
regular_best_50_ep
regular_best_?_ep
regular_initial_20_ep
regular_initial_50_ep
regular_unknown_?_ep
swiss_best_100_ep
swiss_best_100_up
swiss_best_150_ep
swiss_best_150_up
swiss_best_50_ep
swiss_best_50_up
swiss_initial_100_ep
swiss_initial_100_up
swiss_initial_150_ep
swiss_initial_150_up
swiss_initial_50_ep
swiss_initial_50_up

Total loaded: 36


In [27]:
# Purpose: run the next step in the notebook workflow.
des_name = [''
'aggregated_best_20_ep',
'aggregated_best_50_ep',
'aggregated_initial_20_ep',
'aggregated_initial_50_ep',
'regular_best_20_ep',
'regular_best_50_ep',
'regular_initial_20_ep',
'regular_initial_50_ep',]

for name in des_name:
    print(f'{name}: moran mean and sd: {designs[name].moran}, voronoi mean and sd {designs[name].voronoi}')

aggregated_best_20_ep: moran mean and sd: (-0.4307355540534017, 0.07710957212741683), voronoi mean and sd (0.08617360609941198, 0.025368568043779)
aggregated_best_50_ep: moran mean and sd: (-0.6100110574530695, 0.058948479992153614), voronoi mean and sd (0.10735000159963978, 0.01453586953534228)
aggregated_initial_20_ep: moran mean and sd: (-0.22128345642324335, 0.05173158507463266), voronoi mean and sd (0.1046824074094744, 0.04148483698958852)
aggregated_initial_50_ep: moran mean and sd: (-0.3752271602788289, 0.06102138798996939), voronoi mean and sd (0.0984800014674665, 0.020175693446958628)
regular_best_20_ep: moran mean and sd: (-0.3608223938870024, 0.034194354488779374), voronoi mean and sd (0.048519203434214064, 0.023977577693617105)
regular_best_50_ep: moran mean and sd: (-0.4652251017407493, 0.022072577835233715), voronoi mean and sd (0.04067000060603028, 0.009415046538434406)
regular_initial_20_ep: moran mean and sd: (-0.40689883869568744, 0.02710358614800922), voronoi mean an

In [28]:
# Purpose: define helper functions for parsing design names and selecting designs.
# =========================================================
# CELL 5
# NAME HELPERS
# =========================================================

import re

def get_n_from_name(name):
    """Extract sample size n from names such as aggregated_best_50_ep."""
    name = str(name)

    match = re.search(r"_(\d+)_(ep|up)$", name, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))

    raise ValueError(
        f"Could not extract n from design name: {name!r}. "
        "The design name should contain a sample size, e.g. aggregated_best_50_ep."
    )


def get_mode_from_name(name):
    mode = str(name).split("_")[-1]

    if mode not in {"ep", "up"}:
        raise ValueError(f"Unknown mode in design name: {name!r}")

    return mode


def get_pi_column(name):
    mode = get_mode_from_name(name)

    if mode == "ep":
        return "pik_ep"

    if mode == "up":
        return "pik_up"


def get_population_from_name(name, n):
    dfs = popus(n)

    if name.startswith("regular"):
        return dfs["df_RegularPop1000"]

    if name.startswith("aggregated"):
        return dfs["df_AggregatedPop1000"]

    if name.startswith("meuse"):
        return dfs["df_meuse"]

    if name.startswith("swiss"):
        return dfs["df_swiss"]

    raise ValueError(f"Unknown population in design name: {name!r}")


def get_y_columns(df):
    cols = []

    for c in ["z", "plane", "peak", "bird"]:
        if c in df.columns:
            cols.append(c)

    return cols


def selected_design_names():
    names = sorted(designs.keys())

    # Skip unfinished placeholder names such as aggregated_best_?_ep.
    names = [x for x in names if "?" not in x]

    if not INCLUDE_SWISS:
        names = [
            x for x in names
            if not x.startswith("swiss")
        ]

    return names


print("Helpers defined.")
print("Selected designs:", len(selected_design_names()))
print("Selected design names:")
for name in selected_design_names():
    print(" -", name)

Helpers defined.
Selected designs: 20
Selected design names:
 - aggregated_best_20_ep
 - aggregated_best_50_ep
 - aggregated_initial_20_ep
 - aggregated_initial_50_ep
 - meuse_best_10_ep
 - meuse_best_10_up
 - meuse_best_20_ep
 - meuse_best_20_up
 - meuse_best_5_ep
 - meuse_best_5_up
 - meuse_initial_10_ep
 - meuse_initial_10_up
 - meuse_initial_20_ep
 - meuse_initial_20_up
 - meuse_initial_5_ep
 - meuse_initial_5_up
 - regular_best_20_ep
 - regular_best_50_ep
 - regular_initial_20_ep
 - regular_initial_50_ep


In [29]:
# Purpose: run the next step in the notebook workflow.
# =========================================================
# CELL 6
# DESIGN DISTRIBUTION AND TRUE VARIANCE
# =========================================================

def get_samples_probs(design):
    """Return all possible samples and their probabilities."""

    obj = design.all_samples_and_probs

    if callable(obj):
        obj = obj()

    samples, probs = obj

    samples = np.asarray(samples, dtype=int)
    probs = np.asarray(probs, dtype=float)

    probs = probs / probs.sum()

    return samples, probs


def nht_total(sample, y, pi):
    sample = np.asarray(sample, dtype=int)
    return float(np.sum(y[sample] / pi[sample]))


def true_design_variance(samples, probs, y, pi):
    """Exact variance of the NHT estimator under the full design distribution."""

    true_total = float(np.sum(y))

    estimates = np.array([
        nht_total(s, y, pi)
        for s in samples
    ])

    mean_est = float(np.sum(probs * estimates))

    var_true = float(
        np.sum(
            probs * (estimates - true_total)**2
        )
    )

    return var_true, mean_est, true_total

print("True variance functions defined.")

True variance functions defined.


In [30]:
# Purpose: run the next step in the notebook workflow.
# =========================================================
# CELL 7
# SIMPLE LOCAL-MEAN ESTIMATOR
# =========================================================

def get_Dk_simple(i, coords_s, q=8, include_self=False):

    d = np.sqrt(
        np.sum(
            (coords_s - coords_s[i])**2,
            axis=1
        )
    )

    if not include_self:
        d[i] = np.inf

    order = np.argsort(d)

    if include_self:
        neigh = order[:q + 1]
    else:
        neigh = order[:q]

    return neigh, d[neigh]


def get_weights_simple(distances, method="equal"):

    distances = np.asarray(distances, dtype=float)

    if method == "equal":
        w = np.ones_like(distances, dtype=float)

    elif method == "inverse_distance":
        d = distances.copy()
        positive = d[d > 0]
        eps = positive.min() if len(positive) > 0 else 1.0
        d[d == 0] = eps
        w = 1.0 / d

    else:
        raise ValueError("Unknown weight method.")

    w = w / w.sum()

    return w


def var_lm_simple(
    sample,
    y,
    pi,
    coords,
    q=LM_SIMPLE_Q,
    weight_method=LM_SIMPLE_WEIGHT_METHOD,
    include_self=LM_SIMPLE_INCLUDE_SELF
):
    """Simplified local-mean estimator used as an empirical benchmark.

    It uses q nearest sampled neighbours and excludes the unit itself
    by default. This was the version that gave good Meuse-UP results.
    """

    sample = np.asarray(sample, dtype=int)

    coords_s = coords[sample]
    z_s = y[sample] / pi[sample]

    n = len(sample)
    total = 0.0

    for i in range(n):

        D_i, distances = get_Dk_simple(
            i,
            coords_s,
            q=q,
            include_self=include_self
        )

        w = get_weights_simple(
            distances,
            method=weight_method
        )

        local_mean = np.sum(w * z_s[D_i])

        total += (z_s[i] - local_mean)**2

    return float(total)

print("Simple LM defined.")

Simple LM defined.


In [31]:
# Purpose: run the next step in the notebook workflow.
# =========================================================
# CELL 8
# TILLE-STYLE LOCAL-MEAN BENCHMARK
# =========================================================

def sinkhorn_normalize(A, max_iter=200, tol=1e-10):

    W = A.astype(float).copy()
    W[W < 0] = 0.0

    for _ in range(max_iter):

        W_old = W.copy()

        row_sums = W.sum(axis=1)
        row_sums[row_sums == 0] = 1.0
        W = W / row_sums[:, None]

        col_sums = W.sum(axis=0)
        col_sums[col_sums == 0] = 1.0
        W = W / col_sums[None, :]

        if np.max(np.abs(W - W_old)) < tol:
            break

    return W


def get_Dk_tille(i, dist_s, j=3):

    d_i = dist_s[i].copy()
    d_i[i] = np.inf

    nearest_i = np.argsort(d_i)[:j]

    D = set([i])

    for h in nearest_i:

        D.add(int(h))

        d_h = dist_s[h].copy()
        d_h[h] = np.inf

        nearest_h = np.argsort(d_h)[:j]

        for m in nearest_h:
            D.add(int(m))

    return np.array(sorted(D), dtype=int)


def build_W_tille(
    sample,
    pi,
    coords,
    j=TILLE_J,
    distance_power=TILLE_DISTANCE_POWER,
    pi_power=TILLE_PI_POWER,
    include_self=TILLE_INCLUDE_SELF,
    sinkhorn_iter=TILLE_SINKHORN_ITER
):

    sample = np.asarray(sample, dtype=int)

    coords_s = coords[sample]
    pi_s = pi[sample]

    n = len(sample)

    diff = coords_s[:, None, :] - coords_s[None, :, :]
    dist_s = np.sqrt(np.sum(diff**2, axis=2))

    positive = dist_s[dist_s > 0]

    min_pos_dist = positive.min() if len(positive) > 0 else 1.0

    A = np.zeros((n, n), dtype=float)

    for i in range(n):

        D_i = get_Dk_tille(
            i,
            dist_s,
            j=j
        )

        for ell in D_i:

            if (not include_self) and (ell == i):
                continue

            d = dist_s[i, ell]

            if d == 0:
                d = min_pos_dist

            A[i, ell] = (
                1.0 / (d**distance_power)
            ) * (
                1.0 / (pi_s[ell]**pi_power)
            )

    W = sinkhorn_normalize(
        A,
        max_iter=sinkhorn_iter
    )

    return W


def var_lm_tille(
    sample,
    y,
    pi,
    coords,
    j=TILLE_J,
    distance_power=TILLE_DISTANCE_POWER,
    pi_power=TILLE_PI_POWER,
    include_self=TILLE_INCLUDE_SELF,
    sinkhorn_iter=TILLE_SINKHORN_ITER
):
    """Tille-style local-mean benchmark."""

    sample = np.asarray(sample, dtype=int)

    z_s = y[sample] / pi[sample]

    W = build_W_tille(
        sample,
        pi,
        coords,
        j=j,
        distance_power=distance_power,
        pi_power=pi_power,
        include_self=include_self,
        sinkhorn_iter=sinkhorn_iter
    )

    local_mean = W @ z_s

    diff = z_s - local_mean

    v = np.sum(
        W * diff[:, None]**2
    )

    return float(v)

print("Tille-style LM defined.")

Tille-style LM defined.


In [32]:
# Purpose: run the next step in the notebook workflow.
# =========================================================
# CELL 9
# ROBERTSON VarB AND HYBRID VarB
# =========================================================

def var_b_resampling(
    sample,
    all_samples,
    all_probs,
    y,
    pi,
    coords,
    alpha=1.0
):
    """Exact expectation version of Robertson-type resampling estimator.

    alpha = 1:
        original Robertson estimator:
        nearest-neighbour y-imputation, then divide by pi of resampled unit.

    alpha = 0:
        UPB estimator:
        nearest-neighbour z=y/pi imputation.

    alpha = 0.35:
        proposed hybrid:
        alpha * original contribution + (1-alpha) * z contribution.
    """

    sample = np.asarray(sample, dtype=int)

    coords_s = coords[sample]

    y_s = y[sample]
    z_s = y[sample] / pi[sample]

    boot_estimates = []

    for sb in all_samples:

        sb = np.asarray(sb, dtype=int)

        coords_b = coords[sb]

        tau_star = 0.0

        for pos, xb in enumerate(coords_b):

            j = sb[pos]

            d = np.sum(
                (coords_s - xb)**2,
                axis=1
            )

            nearest_position = np.argmin(d)

            raw_part = y_s[nearest_position] / pi[j]

            z_part = z_s[nearest_position]

            tau_star += (
                alpha * raw_part
                +
                (1.0 - alpha) * z_part
            )

        boot_estimates.append(tau_star)

    boot_estimates = np.asarray(boot_estimates, dtype=float)

    mean_boot = np.sum(
        all_probs * boot_estimates
    )

    var_boot = np.sum(
        all_probs * (boot_estimates - mean_boot)**2
    )

    return float(var_boot)


def var_B_original(sample, all_samples, all_probs, y, pi, coords):
    return var_b_resampling(
        sample,
        all_samples,
        all_probs,
        y,
        pi,
        coords,
        alpha=1.0
    )


def var_UPB(sample, all_samples, all_probs, y, pi, coords):
    return var_b_resampling(
        sample,
        all_samples,
        all_probs,
        y,
        pi,
        coords,
        alpha=0.0
    )


def var_B_hybrid_a035(sample, all_samples, all_probs, y, pi, coords):
    return var_b_resampling(
        sample,
        all_samples,
        all_probs,
        y,
        pi,
        coords,
        alpha=HYBRID_ALPHA
    )

print("Robertson VarB and hybrid VarB defined.")

Robertson VarB and hybrid VarB defined.


In [33]:
# Purpose: run the next step in the notebook workflow.
# =========================================================
# CELL 10
# ANALYSE ONE DESIGN AND ONE VARIABLE
# =========================================================

def relbias_from_values(vals, probs, true_var):
    vals = np.asarray(vals, dtype=float)

    mean_val = float(
        np.sum(probs * vals)
    )

    relbias = mean_val / true_var - 1.0

    return mean_val, relbias


def analyse_one_y(design_name, y_col):

    design = designs[design_name]

    n = get_n_from_name(design_name)

    df = get_population_from_name(design_name, n)

    pi_col = get_pi_column(design_name)

    coords = df[["coord_x", "coord_y"]].to_numpy(float)

    y = df[y_col].to_numpy(float)

    pi = df[pi_col].to_numpy(float)

    samples, probs = get_samples_probs(design)

    var_true, mean_nht, true_total = true_design_variance(
        samples,
        probs,
        y,
        pi
    )

    vals_B = []
    vals_UPB = []
    vals_HB = []
    vals_LM = []
    vals_Tille = []

    for s in samples:

        vals_B.append(
            var_B_original(
                s,
                samples,
                probs,
                y,
                pi,
                coords
            )
        )

        vals_UPB.append(
            var_UPB(
                s,
                samples,
                probs,
                y,
                pi,
                coords
            )
        )

        vals_HB.append(
            var_B_hybrid_a035(
                s,
                samples,
                probs,
                y,
                pi,
                coords
            )
        )

        vals_LM.append(
            var_lm_simple(
                s,
                y,
                pi,
                coords
            )
        )

        vals_Tille.append(
            var_lm_tille(
                s,
                y,
                pi,
                coords
            )
        )

    E_B, rb_B = relbias_from_values(vals_B, probs, var_true)
    E_UPB, rb_UPB = relbias_from_values(vals_UPB, probs, var_true)
    E_HB, rb_HB = relbias_from_values(vals_HB, probs, var_true)
    E_LM, rb_LM = relbias_from_values(vals_LM, probs, var_true)
    E_Tille, rb_Tille = relbias_from_values(vals_Tille, probs, var_true)

    return {
        "design": design_name,
        "population": design_name.split("_")[0],
        "design_type": design_name.split("_")[1],
        "n": n,
        "mode": get_mode_from_name(design_name),
        "y": y_col,

        "true_total": true_total,
        "mean_nht": mean_nht,
        "true_var": var_true,

        "E_VarB": E_B,
        "RB_VarB": rb_B,

        "E_UPB": E_UPB,
        "RB_UPB": rb_UPB,

        "E_HybridB_a035": E_HB,
        "RB_HybridB_a035": rb_HB,

        "E_LM_simple_q8": E_LM,
        "RB_LM_simple_q8": rb_LM,

        "E_LM_Tille_j3": E_Tille,
        "RB_LM_Tille_j3": rb_Tille,
    }

print("Analysis function defined.")

Analysis function defined.


In [34]:
# Purpose: save or display the final summary table.
# =========================================================
# CELL 11
# RUN ALL SELECTED DESIGNS
# =========================================================

rows = []

for design_name in selected_design_names():

    print("\n================================================")
    print("Working on:", design_name)
    print("================================================")

    n = get_n_from_name(design_name)

    df = get_population_from_name(design_name, n)

    y_cols = get_y_columns(df)

    for y_col in y_cols:

        print("   y =", y_col)

        row = analyse_one_y(
            design_name,
            y_col
        )

        rows.append(row)

        print(
            "      true_var =",
            round(row["true_var"], 3),
            "| RB VarB =",
            round(100 * row["RB_VarB"], 2),
            "%",
            "| RB UPB =",
            round(100 * row["RB_UPB"], 2),
            "%",
            "| RB Hybrid =",
            round(100 * row["RB_HybridB_a035"], 2),
            "%",
            "| RB LM =",
            round(100 * row["RB_LM_simple_q8"], 2),
            "%",
            "| RB Tille =",
            round(100 * row["RB_LM_Tille_j3"], 2),
            "%"
        )

    if SAVE_PARTIAL:

        partial = pd.DataFrame(rows)

        partial.to_csv(
            OUTPUT_DIR / "variance_results_partial.csv",
            index=False
        )

        print("Saved partial results.")


results = pd.DataFrame(rows)

results.to_csv(
    OUTPUT_DIR / "variance_results_full.csv",
    index=False
)

results


Working on: aggregated_best_20_ep
   y = z
      true_var = 11592.044 | RB VarB = 18.92 % | RB UPB = 18.92 % | RB Hybrid = 18.92 % | RB LM = 170.81 % | RB Tille = 11.2 %
   y = plane
      true_var = 11592.044 | RB VarB = 18.92 % | RB UPB = 18.92 % | RB Hybrid = 18.92 % | RB LM = 170.81 % | RB Tille = 11.2 %
   y = peak
      true_var = 26938.659 | RB VarB = 64.52 % | RB UPB = 64.52 % | RB Hybrid = 64.52 % | RB LM = 581.2 % | RB Tille = 242.39 %
   y = bird
      true_var = 45115.212 | RB VarB = -10.07 % | RB UPB = -10.07 % | RB Hybrid = -10.07 % | RB LM = 372.53 % | RB Tille = 137.64 %
Saved partial results.

Working on: aggregated_best_50_ep
   y = z
      true_var = 3582.873 | RB VarB = 32.06 % | RB UPB = 32.06 % | RB Hybrid = 32.06 % | RB LM = 6.1 % | RB Tille = -63.69 %
   y = plane
      true_var = 3582.873 | RB VarB = 32.06 % | RB UPB = 32.06 % | RB Hybrid = 32.06 % | RB LM = 6.1 % | RB Tille = -63.69 %
   y = peak
      true_var = 5026.759 | RB VarB = 48.64 % | RB UPB = 48.64 

KeyboardInterrupt: 

In [ ]:
# Purpose: save or display the final summary table.
# =========================================================
# CELL 12
# FINAL PAPER TABLE
# =========================================================

paper_results = results[
    [
        "population",
        "design",
        "design_type",
        "n",
        "mode",
        "y",
        "true_var",
        "RB_VarB",
        "RB_UPB",
        "RB_HybridB_a035",
        "RB_LM_simple_q8",
        "RB_LM_Tille_j3",
    ]
].copy()

# Convert relative biases to percentages
for c in [
    "RB_VarB",
    "RB_UPB",
    "RB_HybridB_a035",
    "RB_LM_simple_q8",
    "RB_LM_Tille_j3",
]:
    paper_results[c + "_pct"] =  paper_results[c]

# Keep clean percentage table
paper_results_pct = paper_results[
    [
        "population",
        "design",
        "n",
        "mode",
        "y",
        "true_var",
        "RB_VarB_pct",
        "RB_UPB_pct",
        "RB_HybridB_a035_pct",
        "RB_LM_simple_q8_pct",
        "RB_LM_Tille_j3_pct",
    ]
].copy()

paper_results_pct = paper_results_pct.sort_values(
    ["population", "n", "mode", "design", "y"]
)

paper_results_pct.to_csv(
    OUTPUT_DIR / "paper_variance_table.csv",
    index=False
)

paper_results_pct

In [ ]:
# Purpose: save or display the final summary table.
# =========================================================
# CELL 13
# OPTIONAL: ROUNDED DISPLAY AND LATEX EXPORT
# DECIMAL RELATIVE BIAS SCALE
# =========================================================

# Use decimal-scale results, not percent-scale results
display_table = paper_results.copy()

# ---------------------------------------------------------
# Round true variance
# ---------------------------------------------------------

display_table["true_var"] = display_table["true_var"].round(3)

# ---------------------------------------------------------
# Round relative biases in decimal scale
# Example:
#   0.0807  means  8.07%
#   -0.1558 means -15.58%
# ---------------------------------------------------------

for c in display_table.columns:

    if c.startswith("RB_"):

        display_table[c] = display_table[c].round(4)

# ---------------------------------------------------------
# Save CSV
# ---------------------------------------------------------

display_table.to_csv(
    OUTPUT_DIR / "paper_variance_table_decimal.csv",
    index=False
)

# ---------------------------------------------------------
# Save LaTeX
# ---------------------------------------------------------

latex_table = display_table.to_latex(
    index=False,
    escape=False,
    float_format="%.2f"
)

with open(OUTPUT_DIR / "paper_variance_table_decimal.tex", "w") as f:

    f.write(latex_table)

display_table

In [ ]:
# Purpose: save or display the final summary table.
# =========================================================
# EXTRA CELL: RUN SWISS RESULTS ONLY
# =========================================================

swiss_rows = []

swiss_designs = [
    name for name in sorted(designs.keys())
    if name.startswith("swiss")
]

print("Swiss designs:")
for name in swiss_designs:
    print("  ", name)

for design_name in swiss_designs:

    print("\n================================================")
    print("Working on:", design_name)
    print("================================================")

    n = get_n_from_name(design_name)

    df = get_population_from_name(design_name, n)

    y_cols = get_y_columns(df)

    for y_col in y_cols:

        print("   y =", y_col)

        row = analyse_one_y(
            design_name,
            y_col
        )

        swiss_rows.append(row)

        print(
            "      true_var =",
            round(row["true_var"], 3),
            "| HybridB RB =",
            round(row["RB_HybridB_a035"], 4),
            "| simple LM RB =",
            round(row["RB_LM_simple_q8"], 4),
            "| Tille LM RB =",
            round(row["RB_LM_Tille_j3"], 4)
        )

# =========================================================
# SWISS RESULTS DATAFRAME
# =========================================================

swiss_results = pd.DataFrame(swiss_rows)

swiss_results = swiss_results[
    [
        "population",
        "design",
        "design_type",
        "n",
        "mode",
        "y",
        "true_var",
        "RB_VarB",
        "RB_UPB",
        "RB_HybridB_a035",
        "RB_LM_simple_q8",
        "RB_LM_Tille_j3",
    ]
].copy()

swiss_results.to_csv(
    OUTPUT_DIR / "swiss_variance_results.csv",
    index=False
)

swiss_results

In [ ]:
# Purpose: save or display the final summary table.
# =========================================================
# LATEX TABLE FROM SAVED CSV -- EXACT COLUMN NAMES
# =========================================================

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("variance_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------
# Read saved file
# ---------------------------------------------------------

paper_results = pd.read_csv("paper_variance_table.csv")

# ---------------------------------------------------------
# Keep exactly the columns in your file
# ---------------------------------------------------------

compact_table = paper_results[
    [
        "population",
        "design",
        "n",
        "mode",
        "y",
        "true_var",
        "RB_HybridB_a035_pct",
        "RB_LM_simple_q8_pct",
    ]
].copy()

# ---------------------------------------------------------
# Remove duplicated plane variable
# ---------------------------------------------------------

compact_table = compact_table[
    compact_table["y"] != "plane"
].copy()

# ---------------------------------------------------------
# Simplify design name
# ---------------------------------------------------------

compact_table["design"] = compact_table["design"].apply(
    lambda x: "best" if "_best_" in str(x) else "initial"
)

# ---------------------------------------------------------
# Format true variance in scientific notation
# ---------------------------------------------------------

compact_table["true_var"] = compact_table["true_var"].map(
    lambda x: f"{x:.2e}"
)

# ---------------------------------------------------------
# Relative bias: decimal scale, rounded to two decimals
# ---------------------------------------------------------

compact_table["RB_HybridB_a035_pct"] = compact_table["RB_HybridB_a035_pct"].map(
    lambda x: f"{x:.2f}"
)

compact_table["RB_LM_simple_q8_pct"] = compact_table["RB_LM_simple_q8_pct"].map(
    lambda x: f"{x:.2f}"
)

# ---------------------------------------------------------
# Rename for LaTeX
# ---------------------------------------------------------

compact_table = compact_table.rename(
    columns={
        "population": "Population",
        "design": "Design",
        "n": "$n$",
        "mode": "Mode",
        "y": "Variable",
        "true_var": "$V_p(\\widehat Y_{\\mathrm{NHT}})$",
        "RB_HybridB_a035_pct": "$\\mathrm{RB}(\\widehat V_{B,0.35})$",
        "RB_LM_simple_q8_pct": "$\\mathrm{RB}(\\widehat V_{\\mathrm{LM},8})$",
    }
)

# ---------------------------------------------------------
# Export LaTeX
# ---------------------------------------------------------

latex_table = compact_table.to_latex(
    index=False,
    escape=False,
    column_format="llcclcc"
)

print(latex_table)

with open(OUTPUT_DIR / "compact_variance_table.tex", "w") as f:
    f.write(latex_table)

compact_table